# Hyperparameter Optimization for Support Vector Classification

This notebook compares three reproducible hyperparameter-search strategies:

1. **Grid search**
2. **Randomized search**
3. **Bayesian search**

The lesson uses scikit-learn's built-in Iris dataset, so no external CSV file or internet download is required.

## Learning objectives

After completing this notebook, students should be able to:

- distinguish model parameters from hyperparameters;
- construct a preprocessing-and-model pipeline;
- avoid data leakage during scaling;
- define appropriate search spaces;
- compare search strategies using cross-validation;
- evaluate the selected model on an unseen test set.

## 1. Import the required libraries

`scikit-optimize` must be installed through the repository-level `requirements.txt` file.

In [ ]:
import time

import numpy as np
import pandas as pd
from scipy.stats import loguniform
from sklearn.datasets import load_iris
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score,
)
from sklearn.model_selection import (
    GridSearchCV,
    RandomizedSearchCV,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from skopt import BayesSearchCV
from skopt.space import Categorical, Real

RANDOM_STATE = 42

## 2. Load and inspect the dataset

The Iris dataset contains four numerical measurements and three flower classes.

In [ ]:
iris = load_iris(as_frame=True)

X = iris.data
y = iris.target

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)
print("Class names:", list(iris.target_names))

X.head()

## 3. Create a stratified train-test split

The test set is held back from all hyperparameter searches. Stratification preserves approximately the same class distribution in both subsets.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,
)

print("Training samples:", len(X_train))
print("Test samples:", len(X_test))
print("\nTraining class counts:")
print(y_train.value_counts().sort_index())

## 4. Build a leakage-safe pipeline

Scaling is performed **inside** each cross-validation fold. This prevents information from validation data from influencing the scaler fitted for a training fold.

In [ ]:
pipeline = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        ("svc", SVC()),
    ]
)

SCORING = "f1_macro"
CV_FOLDS = 5

## 5. Grid search

Grid search evaluates every explicitly listed combination. It is systematic, but the number of model fits grows quickly as the grid expands.

In [ ]:
grid_parameters = {
    "svc__C": [0.1, 1, 10, 100],
    "svc__gamma": ["scale", 0.01, 0.1, 1],
    "svc__kernel": ["linear", "rbf"],
}

grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=grid_parameters,
    scoring=SCORING,
    cv=CV_FOLDS,
    n_jobs=-1,
    return_train_score=True,
)

grid_start = time.perf_counter()
grid_search.fit(X_train, y_train)
grid_duration = time.perf_counter() - grid_start

print("Best parameters:", grid_search.best_params_)
print(f"Best cross-validation F1-macro: {grid_search.best_score_:.4f}")
print(f"Search time: {grid_duration:.3f} seconds")

## 6. Randomized search

Randomized search samples a fixed number of combinations from probability distributions. It can explore a wider continuous space with a controlled computational budget.

In [ ]:
random_parameters = {
    "svc__C": loguniform(1e-3, 1e3),
    "svc__gamma": loguniform(1e-4, 1e1),
    "svc__kernel": ["linear", "rbf"],
}

random_search = RandomizedSearchCV(
    estimator=pipeline,
    param_distributions=random_parameters,
    n_iter=25,
    scoring=SCORING,
    cv=CV_FOLDS,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    return_train_score=True,
)

random_start = time.perf_counter()
random_search.fit(X_train, y_train)
random_duration = time.perf_counter() - random_start

print("Best parameters:", random_search.best_params_)
print(f"Best cross-validation F1-macro: {random_search.best_score_:.4f}")
print(f"Search time: {random_duration:.3f} seconds")

## 7. Bayesian search

Bayesian optimization uses results from earlier evaluations to select promising new combinations. This can be more sample-efficient than uninformed random sampling.

In [ ]:
bayesian_parameters = {
    "svc__C": Real(1e-3, 1e3, prior="log-uniform"),
    "svc__gamma": Real(1e-4, 1e1, prior="log-uniform"),
    "svc__kernel": Categorical(["linear", "rbf"]),
}

bayesian_search = BayesSearchCV(
    estimator=pipeline,
    search_spaces=bayesian_parameters,
    n_iter=25,
    scoring=SCORING,
    cv=CV_FOLDS,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    return_train_score=True,
)

bayesian_start = time.perf_counter()
bayesian_search.fit(X_train, y_train)
bayesian_duration = time.perf_counter() - bayesian_start

print("Best parameters:", bayesian_search.best_params_)
print(
    "Best cross-validation F1-macro:",
    f"{bayesian_search.best_score_:.4f}",
)
print(f"Search time: {bayesian_duration:.3f} seconds")

## 8. Compare the search strategies

Cross-validation scores estimate performance using the training data. Search time depends on the machine, available CPU cores, library versions, and the specific sampled candidates.

In [ ]:
comparison = pd.DataFrame(
    [
        {
            "method": "Grid search",
            "candidates_evaluated": len(grid_search.cv_results_["params"]),
            "best_cv_f1_macro": grid_search.best_score_,
            "search_seconds": grid_duration,
        },
        {
            "method": "Randomized search",
            "candidates_evaluated": len(random_search.cv_results_["params"]),
            "best_cv_f1_macro": random_search.best_score_,
            "search_seconds": random_duration,
        },
        {
            "method": "Bayesian search",
            "candidates_evaluated": len(
                bayesian_search.cv_results_["params"]
            ),
            "best_cv_f1_macro": bayesian_search.best_score_,
            "search_seconds": bayesian_duration,
        },
    ]
)

comparison.round(
    {
        "best_cv_f1_macro": 4,
        "search_seconds": 3,
    }
)

## 9. Evaluate each selected model on the held-out test set

The test set is used only after the searches have completed. The conventional metric argument order is:

```python
metric(y_true, y_pred)
```

In [ ]:
searches = {
    "Grid search": grid_search,
    "Randomized search": random_search,
    "Bayesian search": bayesian_search,
}

test_results = []

for method, search in searches.items():
    predictions = search.best_estimator_.predict(X_test)

    test_results.append(
        {
            "method": method,
            "test_accuracy": accuracy_score(y_test, predictions),
            "test_f1_macro": f1_score(
                y_test,
                predictions,
                average="macro",
            ),
        }
    )

pd.DataFrame(test_results).round(4)

## 10. Inspect the final classification report

The method with the strongest cross-validation score is selected without looking at the test labels.

In [ ]:
best_method, best_search = max(
    searches.items(),
    key=lambda item: item[1].best_score_,
)

final_predictions = best_search.best_estimator_.predict(X_test)

print("Selected method:", best_method)
print("Selected parameters:", best_search.best_params_)
print()
print(
    classification_report(
        y_test,
        final_predictions,
        target_names=iris.target_names,
    )
)

## Key observations

- **Grid search** is easy to interpret but can become expensive for large grids.
- **Randomized search** explores continuous spaces with a fixed evaluation budget.
- **Bayesian search** adapts later trials based on earlier observations.
- The strongest cross-validation score does not guarantee the strongest test score.
- Preprocessing must be included inside the pipeline to avoid data leakage.
- Hyperparameters should come from `best_params_` or `best_estimator_`, not from manually copied values.
- The held-out test set must not guide hyperparameter selection.

## Exercises

1. Increase `n_iter` for randomized and Bayesian search.
2. Replace F1-macro with accuracy and compare the selected parameters.
3. Add the polynomial SVC kernel.
4. Add `class_weight` to the search space.
5. Replace Iris with another built-in classification dataset.
6. Compare the variability of results across several random seeds.
7. Explain why the test set must remain outside the search procedure.